In [3]:
"""
Data analysis of BraTS OS scores to understand their distribution.
Expects a CSV like the following:
Brats20ID,Age,Survival_days,Extent_of_Resection
BraTS20_Training_001,60.463,289,GTR
"""
import pandas as pd

def quantile_bin_boundaries(values: pd.Series, x_bins: int) -> list[float]:
    """Return quantile-based bin edges for `values`.

    The function targets approximately equal-frequency bins (about N / x_bins values
    per bin) after dropping NaNs.
    """
    if x_bins < 1:
        raise ValueError("x_bins must be >= 1")

    clean = pd.to_numeric(values, errors="coerce").dropna()
    if clean.empty:
        raise ValueError("No valid numeric values found for binning")

    # qcut computes equal-frequency bins and returns the edges via retbins=True.
    # duplicates='drop' handles repeated quantile edges in low-variance data.
    _, edges = pd.qcut(clean, q=x_bins, retbins=True, duplicates="drop")
    edges_formatted = [float(f"{edge:.2f}") for edge in edges]
    
    return edges_formatted


csv_path = "/path/to/BrainWear_Kareem/BraTS_OS.csv"
df = pd.read_csv(csv_path)

# Ensure numeric columns are treated as numeric
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Survival_days"] = pd.to_numeric(df["Survival_days"], errors="coerce")

# Min/max stats for numeric columns
for col in ["Age", "Survival_days"]:
    print(f"{col}: min={df[col].min()}, max={df[col].max()}")

# Summary statistics
print("\nSummary statistics:")
print(df[["Age", "Survival_days"]].describe())

# Extent of resection counts
print("\nExtent_of_Resection counts:")
print(df["Extent_of_Resection"].value_counts(dropna=False))

# Example: split Survival_days into X quantile bins and return boundaries
X = 5
boundaries = quantile_bin_boundaries(df["Survival_days"], x_bins=X)
print(f"\nQuantile bin boundaries for Survival_days (X={X}):")
print(boundaries)

# Count how many values fall into each bin defined by the returned boundaries
survival_bins = pd.cut(df["Survival_days"], bins=boundaries, include_lowest=True)
bin_counts = survival_bins.value_counts(sort=False)
print("\nCounts per bin:")
for interval, count in bin_counts.items():
    print(f"{interval}: {count}")

